# <font color = "red">**Hackathon Challenge — Deciphering the Decision Logic of an AI Agent**</font>

- Can we infer what an AI agent values by observing the sequence of decisions it makes?

- In this challenge, GPT acts as an autonomous microscopy agent that repeatedly selects one measurement location from a set of candidate AFM image patches. 

- We record its decisions and then attempt to reverse-engineer the hidden logic behind those choices using visual, statistical, and data-driven methods.

<font color = "blue" size = 6>**Goal**</font>: develop a method that best explains GPT’s behavior and predicts its future decisions on unseen candidate sets.

<a href="https://colab.research.google.com/github/kbarakati/camm_hackathon/blob/k4my4r/docs/day_20_10092026/llm_decision_making.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import aespm as ae
from pathlib import Path

# Machine-learning tools we will use later
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

## load data

In [ ]:
fname = r"scan_main_0000.ibw"

data = ae.tools.load_ibw(fname)

print("Channels:", data.channels)
print("Shape:", data.data.shape)
print("Mode:", data.mode)

plot the AFM image

In [ ]:
channel_index = 2

img = data.data[channel_index]

img = (img - img.min()) / (img.max() - img.min())

plt.figure(figsize=(5, 5))
plt.imshow(img, cmap="gray")
plt.axis("off")
plt.show()

## Generate measurement locations

At each experimental step, the **agent** should not search the entire image pixel-by-pixel. Instead, we give it 10 possible locations and ask:

Which of these 10 regions should I measure next?

In [ ]:
N_CANDIDATES = 10
PATCH_SIZE = 16

rng = np.random.default_rng(42)

H, W = img.shape
half = PATCH_SIZE // 2

# Random patch centers
candidate_centers = [
    (
        rng.integers(half, H - half),
        rng.integers(half, W - half)
    )
    for _ in range(N_CANDIDATES)
]

# Extract patches
candidate_patches = [
    img[y-half:y+half, x-half:x+half]
    for y, x in candidate_centers
]

# Display
fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for i, (ax, patch) in enumerate(zip(axes.flat, candidate_patches)):
    ax.imshow(patch, cmap="gray")
    ax.set_title(f"Candidate {i}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## Extract simple interpretable features

In [ ]:
# ============================================================
# BLOCK 3 — EXTRACT AND NORMALIZE PATCH FEATURES
# ============================================================

features = []

for patch in candidate_patches:

    mean_intensity = np.mean(patch)
    contrast = np.std(patch)

    gy, gx = np.gradient(patch.astype(float))
    edge_strength = np.mean(np.sqrt(gx**2 + gy**2))

    features.append([mean_intensity, contrast, edge_strength])


features = pd.DataFrame(
    features,
    columns=["mean_intensity", "contrast", "edge_strength"]
)

# # Normalize every feature between 0 and 1
features = (features - features.min()) / (features.max() - features.min())

features.round(3)

## agent’s hidden scientific objective

In [ ]:
# ============================================================
# BLOCK 5 — HIDDEN REWARD FUNCTION
# ============================================================

weights = {
    "mean_intensity": 0.10,
    "contrast":       0.35,
    "edge_strength":  0.55
}

reward = (
    weights["mean_intensity"] * features["mean_intensity"] +
    weights["contrast"]       * features["contrast"] +
    weights["edge_strength"]  * features["edge_strength"]
)

reward

## Let the agent choose one candidate

In [ ]:
selected_idx = reward.idxmax()

print(f"Agent selected Candidate {selected_idx}")
print(f"Reward = {reward[selected_idx]:.3f}")

which patch

In [ ]:
plt.figure(figsize=(3, 3))

plt.imshow(candidate_patches[selected_idx], cmap="gray")
plt.title(f"Selected Candidate {selected_idx}")
plt.axis("off")

plt.show()

What happened here?
The agent evaluated all 10 candidates using its hidden objective:

features --> reward --> selection

## Generate a full decision history

In [ ]:
N_STEPS = 30
history = []

for step in range(N_STEPS):

    # Generate 10 new candidate locations
    centers = [
        (
            rng.integers(half, H - half),
            rng.integers(half, W - half)
        )
        for _ in range(N_CANDIDATES)
    ]

    # Extract patches
    patches = [
        img[y-half:y+half, x-half:x+half]
        for y, x in centers
    ]

    # Compute features
    step_features = []

    for patch in patches:
        mean_intensity = np.mean(patch)
        contrast = np.std(patch)

        gy, gx = np.gradient(patch)
        edge_strength = np.mean(np.sqrt(gx**2 + gy**2))

        step_features.append([
            mean_intensity,
            contrast,
            edge_strength
        ])

    step_features = pd.DataFrame(
        step_features,
        columns=["mean_intensity", "contrast", "edge_strength"]
    )

    # Normalize features within this decision step
    fmin = step_features.min()
    fmax = step_features.max()

    step_features = (step_features - fmin) / (fmax - fmin + 1e-12)

    # Hidden reward
    step_reward = (
        weights["mean_intensity"] * step_features["mean_intensity"] +
        weights["contrast"]       * step_features["contrast"] +
        weights["edge_strength"]  * step_features["edge_strength"]
    )

    # Agent chooses highest-reward candidate
    selected = step_reward.idxmax()

    # Save all 10 candidates
    for i, (y, x) in enumerate(centers):

        history.append({
            "step": step,
            "candidate": i,
            "x": x,
            "y": y,
            "mean_intensity": step_features.loc[i, "mean_intensity"],
            "contrast": step_features.loc[i, "contrast"],
            "edge_strength": step_features.loc[i, "edge_strength"],
            "reward_true": step_reward[i],
            "selected": int(i == selected)
        })


history = pd.DataFrame(history)

history.head(20)

## History of decision on the map

In [ ]:
selected_history = history[history["selected"] == 1]

plt.figure(figsize=(6, 6))

plt.imshow(img, cmap="gray")

plt.scatter(
    selected_history["x"],
    selected_history["y"],
    s=40,
    facecolors="r",
    edgecolors="r",

    marker = "o"
)

plt.axis("off")
plt.show()

first 4 decison been among these options

In [ ]:
# ============================================================
# PLOT FIRST 4 DECISION STEPS (2x2 GRID)
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for ax, step in zip(axes.flat, range(4)):

    step_data = history[history["step"] == step]
    selected = step_data[step_data["selected"] == 1]
    not_selected = step_data[step_data["selected"] == 0]

    ax.imshow(img, cmap="gray")

    # 9 not selected
    ax.scatter(
        not_selected["x"],
        not_selected["y"],
        s=40,
        marker="s",
        facecolors="none",
        edgecolors="lime"
    )

    # 1 selected
    ax.scatter(
        selected["x"],
        selected["y"],
        s=60,
        marker="o",
        facecolors="none",
        edgecolors="red"
    )

    ax.set_title(f"Step {step}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## decipher the hidden reward

Compare what the agent selects vs rejects

In [ ]:
feature_cols = [
    "mean_intensity",
    "contrast",
    "edge_strength"
]

comparison = history.groupby("selected")[feature_cols].mean()

comparison.index = ["Not selected", "Selected"]

comparison.round(3)

fit a model that tries to predict the agent's choices from the features

In [ ]:
X = history[feature_cols]
y = history["selected"]

model = LogisticRegression()
model.fit(X, y)

learned_weights = pd.Series(
    model.coef_[0],
    index=feature_cols
)

learned_weights

In [ ]:
importance = np.abs(learned_weights)

importance = importance / importance.sum()

importance = importance.sort_values(ascending=False)

importance.round(3)

In [ ]:
plt.figure(figsize=(6, 4))

importance.plot(kind="bar")

plt.ylabel("Relative importance")
plt.title("Inferred Agent Objective")
plt.ylim(0, 1)
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

Compare inferred weights with the true hidden reward

In [ ]:
true_weights = pd.Series(weights)

comparison_weights = pd.DataFrame({
    "True": true_weights,
    "Inferred": importance
})

comparison_weights

In [ ]:
comparison_weights.plot(
    kind="bar",
    figsize=(7, 4)
)

plt.ylabel("Relative importance")
plt.title("True vs Inferred Agent Objective")
plt.xticks(rotation=0)
plt.ylim(0, 1)

plt.tight_layout()
plt.show()

# Real LLM decision making

Previously, we defined the reward and the simulated agent followed it.

Now, GPT will make the decisions itself, and we will observe its choices and try to infer what implicit criteria explain them.

In [ ]:
# %pip install -q -U openai httpx2 "brotli>=1.2.0"

In [ ]:
from openai import OpenAI

# Replace with your API key
OPENAI_API_KEY = "API key here"

client = OpenAI(api_key=OPENAI_API_KEY)

MODEL = "gpt-5.6-sol"

print("GPT client ready")

## Convert candidate patches to PNG/base64

In [ ]:
import io
import base64
from PIL import Image

def patch_to_base64(patch):

    # Convert normalized 0–1 image to 8-bit grayscale
    patch_uint8 = (patch * 255).clip(0, 255).astype(np.uint8)

    image = Image.fromarray(patch_uint8)

    buffer = io.BytesIO()
    image.save(buffer, format="PNG")

    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")

    return f"data:image/png;base64,{encoded}"


candidate_images = [
    patch_to_base64(patch)
    for patch in candidate_patches
]

print(f"Prepared {len(candidate_images)} candidate images for GPT.")

## Ask GPT to make one real decision

define prompt

In [ ]:
content = [
    {
        "type": "input_text",
        "text": """
You are controlling an autonomous microscopy experiment.

You are given 10 candidate image patches from possible measurement locations.

Select the ONE candidate that you believe would be the most scientifically
informative location to measure next.

Base your decision only on the visual information in the patches.

Return ONLY the candidate number from 0 to 9.
"""
    }
]

for i, image in enumerate(candidate_images):

    content.append({
        "type": "input_text",
        "text": f"Candidate {i}"
    })

    content.append({
        "type": "input_image",
        "image_url": image,
        "detail": "high"
    })

## make the actual API call

In [ ]:
response = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "user",
            "content": content
        }
    ],
    extra_headers={
        "Accept-Encoding": "identity"
    }
)

gpt_choice = int(response.output_text.strip())

print(f"GPT selected Candidate {gpt_choice}")

visualize the decision

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for i, (ax, patch) in enumerate(zip(axes.flat, candidate_patches)):

    ax.imshow(patch, cmap="gray")

    if i == gpt_choice:
        ax.set_title(f"Candidate {i} ✓", color="red")
        for spine in ax.spines.values():
            spine.set_edgecolor("red")
            spine.set_linewidth(3)
    else:
        ax.set_title(f"Candidate {i}")

    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

test gpt decision on same set to check if it is not random

In [ ]:
from collections import Counter

N_REPEATS = 3
choices = []

for i in range(N_REPEATS):

    response = client.responses.create(
        model=MODEL,
        input=[{
            "role": "user",
            "content": content
        }],
        extra_headers={
            "Accept-Encoding": "identity"
        }
    )

    choice = int(response.output_text.strip())
    choices.append(choice)

    print(f"Run {i+1:02d} → Candidate {choice}")

In [ ]:
counts = Counter(choices)

most_common_choice, n_common = counts.most_common(1)[0]

consistency = n_common / N_REPEATS

print(f"\nMost selected candidate: {most_common_choice}")
print(f"Consistency: {consistency:.1%}")

## Build a GPT decision history

In [ ]:
# ============================================================
# REAL LLM — HELPER FUNCTIONS
# ============================================================

def get_candidates(img, rng):

    centers = [
        (
            rng.integers(half, H - half),
            rng.integers(half, W - half)
        )
        for _ in range(N_CANDIDATES)
    ]

    patches = [
        img[y-half:y+half, x-half:x+half]
        for y, x in centers
    ]

    return centers, patches

def ask_gpt(patches):

    images = [patch_to_base64(patch) for patch in patches]

    content = [{
        "type": "input_text",
        "text": """
You are controlling an autonomous microscopy experiment.

Choose the ONE candidate that is the most scientifically
informative location to measure next.

Use only the visual information.

Return ONLY one integer from 0 to 9.
"""
    }]

    for i, image in enumerate(images):

        content += [
            {"type": "input_text", "text": f"Candidate {i}"},
            {
                "type": "input_image",
                "image_url": image,
                "detail": "high"
            }
        ]

    response = client.responses.create(
        model=MODEL,
        input=[{"role": "user", "content": content}],
        extra_headers={"Accept-Encoding": "identity"}
    )

    return int(response.output_text.strip())

In [ ]:
# ============================================================
# REAL LLM — COLLECT GPT DECISIONS
# ============================================================

N_GPT_STEPS = 20
gpt_rng = np.random.default_rng(123)

gpt_history = []

for step in range(N_GPT_STEPS):

    centers, patches = get_candidates(img, gpt_rng)

    selected = ask_gpt(patches)

    for i, (y, x) in enumerate(centers):

        gpt_history.append({
            "step": step,
            "candidate": i,
            "x": x,
            "y": y,
            "selected": int(i == selected)
        })

    print(f"Step {step + 1:02d} → Candidate {selected}")


gpt_history = pd.DataFrame(gpt_history)

In [ ]:
selected_gpt = gpt_history[gpt_history["selected"] == 1]

plt.figure(figsize=(6, 6))
plt.imshow(img, cmap="gray")

plt.scatter(
    selected_gpt["x"],
    selected_gpt["y"],
    s=40,
    marker="o",
    facecolors="red",
    edgecolors="red"
)

plt.axis("off")
plt.show()

## Reconstruct the decision episodes

In [ ]:
# ============================================================
# REVERSE ENGINEERING — BLOCK 1: RECONSTRUCT DECISION EPISODES
# ============================================================

decision_history = []

for step in sorted(gpt_history["step"].unique()):

    step_data = gpt_history[gpt_history["step"] == step]

    patches = [
        img[
            int(row.y)-half:int(row.y)+half,
            int(row.x)-half:int(row.x)+half
        ]
        for _, row in step_data.iterrows()
    ]

    selected = int(
        step_data.loc[step_data["selected"] == 1, "candidate"].iloc[0]
    )

    decision_history.append({
        "step": step,
        "patches": patches,
        "selected": selected
    })

print(f"Reconstructed {len(decision_history)} GPT decisions.")

## Build the patch dataset

In [ ]:
X_patches = []
y_selected = []
step_ids = []
candidate_ids = []

for _, row in gpt_history.iterrows():

    x = int(row["x"])
    y = int(row["y"])

    patch = img[
        y-half:y+half,
        x-half:x+half
    ]

    X_patches.append(patch)
    y_selected.append(row["selected"])
    step_ids.append(row["step"])
    candidate_ids.append(row["candidate"])


X_patches = np.array(X_patches)
y_selected = np.array(y_selected)

print("Patch dataset:", X_patches.shape)
print("Selected:", y_selected.sum())
print("Not selected:", len(y_selected) - y_selected.sum())

## Approach 1 — Visual Deciphering

In [ ]:
from sklearn.decomposition import PCA

# Flatten image patches
X = X_patches.reshape(len(X_patches), -1)

# Reduce to 2 dimensions
X_pca = PCA(n_components=2).fit_transform(X)

# Plot
plt.figure(figsize=(6, 5))

plt.scatter(
    X_pca[y_selected == 0, 0],
    X_pca[y_selected == 0, 1],
    alpha=0.4,
    label="Not selected"
)

plt.scatter(
    X_pca[y_selected == 1, 0],
    X_pca[y_selected == 1, 1],
    s=60,
    label="GPT selected"
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Visual Structure of GPT Decisions")
plt.legend()
plt.show()

**Question**

Do GPT-selected patches occupy a distinct region of image space?

- If yes, GPT may have a consistent visual preference.

- If not, the decision rule may be more complex than the first two PCA dimensions can reveal.

In [ ]:
# ============================================================
# TRAIN / TEST SPLIT BY DECISION STEP
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

step_ids = np.array(step_ids)

train_steps, test_steps = train_test_split(
    np.unique(step_ids),
    test_size=0.25,
    random_state=42
)

train_mask = np.isin(step_ids, train_steps)
test_mask  = np.isin(step_ids, test_steps)


def decision_accuracy(scores, labels, steps):

    correct = []

    for step in np.unique(steps):

        mask = steps == step

        predicted = np.argmax(scores[mask])
        actual    = np.argmax(labels[mask])

        correct.append(predicted == actual)

    return np.mean(correct)

Can PCA predict GPT's decision?

In [ ]:
pca = PCA(n_components=2)

X_train = pca.fit_transform(X[train_mask])
X_test  = pca.transform(X[test_mask])

model = LinearRegression()
model.fit(X_train, y_selected[train_mask])

scores = model.predict(X_test)

accuracy = decision_accuracy(
    scores,
    y_selected[test_mask],
    step_ids[test_mask]
)

print(f"PCA accuracy:    {accuracy:.1%}")
print(f"Random baseline: {1/N_CANDIDATES:.1%}")

## Approach 2 — Statistical Deciphering

In [ ]:
from scipy.stats import skew, kurtosis, mannwhitneyu

stats = []

for patch in X_patches:

    p = patch.ravel()

    stats.append({
        "mean": np.mean(p),
        "std": np.std(p),
        "range": np.ptp(p),
        "skewness": skew(p),
        "kurtosis": kurtosis(p)
    })

stats_df = pd.DataFrame(stats)
stats_df["selected"] = y_selected

In [ ]:
results = []

for feature in stats_df.columns[:-1]:

    selected = stats_df[stats_df["selected"] == 1][feature]
    rejected = stats_df[stats_df["selected"] == 0][feature]

    U, p = mannwhitneyu(selected, rejected)

    effect = 2 * U / (len(selected) * len(rejected)) - 1

    results.append([
        feature,
        effect,
        p
    ])

results = pd.DataFrame(
    results,
    columns=["feature", "effect_size", "p_value"]
)

results = results.sort_values(
    "effect_size",
    key=np.abs,
    ascending=False
)

results

In [ ]:
plt.figure(figsize=(6, 4))

plt.barh(
    results["feature"],
    results["effect_size"]
)

plt.axvline(0, color="black")
plt.xlabel("Effect size")
plt.title("Statistics Associated with GPT Selection")

plt.show()

Can the statistics predict GPT's decision?

In [ ]:
X_stats = stats_df.drop(columns="selected").values

model = LinearRegression()

model.fit(
    X_stats[train_mask],
    y_selected[train_mask]
)

scores = model.predict(
    X_stats[test_mask]
)

accuracy = decision_accuracy(
    scores,
    y_selected[test_mask],
    step_ids[test_mask]
)

print(f"Statistics accuracy: {accuracy:.1%}")
print(f"Random baseline:     {1/N_CANDIDATES:.1%}")

# Can you develop a better way to decipher GPT and achieve higher prediction accuracy?

In [ ]:
# ============================================================
# CHALLENGE — DECIPHER GPT'S DECISION LOGIC
# ============================================================

# Can you find a representation of the image patches
# that better explains GPT's decisions?
#
# Your goal:
#
# 1. Develop your own representation / features
# 2. Train a model using the training decisions
# 3. Predict GPT's choices on the test decisions
# 4. Compare your accuracy against:
#
#       Random baseline
#       PCA approach
#       Statistical approach


# ============================================================
# YOUR CODE STARTS HERE
# ============================================================






# ============================================================
# YOUR CODE ENDS HERE
# ============================================================